# GHArchive — Datalab Demo

Reads raw GitHub Archive data from Apache Ozone, queries it with Spark, and writes an Iceberg table.

**Catalog**: `iceberg` (HadoopCatalog, warehouse = `s3a://curated/`) — no Hive metastore needed.

## 1. Verify S3A connectivity

In [ ]:
import os
import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(s3={"addressing_style": "path"}),
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print("Buckets:", buckets)

## 2. Spark — read raw GHArchive data

`spark-defaults.conf` pre-configures S3A and the `iceberg` catalog — no extra setup needed.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("GHArchive Datalab").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("S3A endpoint:", spark.conf.get("spark.hadoop.fs.s3a.endpoint"))
print("Iceberg warehouse:", spark.conf.get("spark.sql.catalog.iceberg.warehouse"))

In [ ]:
raw_path = "s3a://raw/gh_archive/"

df = spark.read.json(raw_path)
print(f"Schema ({df.count()} rows):")
df.printSchema()

In [ ]:
# Top event types
df.groupBy("type").count().orderBy("count", ascending=False).show(10)

## 3. Write Iceberg table to curated bucket

Table is written via the `iceberg` catalog (HadoopCatalog).  
Metadata is stored at `s3a://curated/curated/github_events/` — no Hive metastore required.

In [ ]:
from pyspark.sql.functions import col, year, month, dayofmonth, to_timestamp

curated = (
    df.select(
        col("id"),
        col("type"),
        col("actor.login").alias("actor"),
        col("repo.name").alias("repo"),
        to_timestamp(col("created_at")).alias("created_at"),
    )
    .withColumn("year",  year("created_at"))
    .withColumn("month", month("created_at"))
    .withColumn("day",   dayofmonth("created_at"))
)

curated.printSchema()

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.curated")

(
    curated.writeTo("iceberg.curated.github_events")
    .partitionedBy("year", "month", "day")
    .createOrReplace()
)

print("Table written to s3a://curated/curated/github_events")

## 4. Read back with Spark

In [ ]:
spark.read.table("iceberg.curated.github_events") \
    .groupBy("type").count() \
    .orderBy("count", ascending=False) \
    .show(10)

## 5. Query back via PyIceberg

`StaticTable.from_metadata()` reads the Iceberg metadata file directly from Ozone —
no catalog server needed. Works with any table written by Spark's HadoopCatalog.

In [ ]:
from pyiceberg.table import StaticTable

ICEBERG_PROPS = {
    "s3.endpoint": os.environ["AWS_S3_ENDPOINT"],
    "s3.access-key-id": os.environ["AWS_ACCESS_KEY_ID"],
    "s3.secret-access-key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "s3.path-style-access": "true",
}

# Locate the latest metadata file written by Spark HadoopCatalog
# HadoopCatalog stores tables at: {warehouse}/{namespace}/{table}/metadata/
metadata_prefix = "curated/github_events/metadata/"
objects = s3.list_objects_v2(Bucket="curated", Prefix=metadata_prefix)
metadata_files = sorted(
    [o["Key"] for o in objects.get("Contents", []) if o["Key"].endswith(".metadata.json")]
)

if not metadata_files:
    raise FileNotFoundError(f"No Iceberg metadata found under s3://curated/{metadata_prefix}")

latest = f"s3://curated/{metadata_files[-1]}"
print("Loading from:", latest)

table = StaticTable.from_metadata(latest, properties=ICEBERG_PROPS)
print(table.schema())

In [ ]:
df_arrow = table.scan(limit=100).to_arrow()
df_arrow.to_pandas().head(10)